# Generalized Linear Models & GDA

[← Back to lesson](https://ml-viz-ruby.vercel.app/courses/linear-regression/04-generalized-linear-models)

This notebook implements three GLMs from scratch (linear regression, logistic regression, Poisson regression), visualizes soft-label distributions, and fits Gaussian Discriminant Analysis — then compares its decision boundary to logistic regression on the same dataset.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor': '#1a1d27',
    'text.color': '#e2e8f0',
    'axes.labelcolor': '#94a3b8',
    'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8',
    'axes.edgecolor': '#2d3748',
    'grid.color': '#2d3748',
    'axes.grid': True,
})

np.random.seed(42)

## Intuition — one framework, many regressions

Linear and logistic regression look different, but they're two instances of one idea: the
**Generalized Linear Model**. A GLM keeps the linear predictor `η = w·x` but feeds it through a
**link function** matched to the response's distribution — **identity** for continuous targets
(Gaussian → linear regression), **logit/sigmoid** for binary (Bernoulli → logistic), **log/exp** for
counts (Poisson → Poisson regression). The remarkable payoff: **the gradient is the same** for all of
them, `Xᵀ(y − ŷ)`. We also meet **GDA** (Gaussian Discriminant Analysis), a *generative* cousin that
models each class as a Gaussian — and see how it relates to logistic regression. Everything is
validated against `sklearn`.

## 1. The exponential family

We show that Bernoulli, Gaussian, and Poisson distributions all fit the exponential family form:
$$p(y;\eta) = b(y)\exp(\eta T(y) - a(\eta))$$

In [ ]:
def sigmoid(z): return 1 / (1 + np.exp(-z))

# Show the natural parameter η maps to different link functions
eta = np.linspace(-4, 4, 200)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Bernoulli: η = log(p/1-p), so p = σ(η)
axes[0].plot(eta, sigmoid(eta), color='#6366f1', linewidth=2)
axes[0].set_title('Bernoulli: p = σ(η)', color='#e2e8f0')
axes[0].set_xlabel('Natural parameter η')
axes[0].set_ylabel('Mean E[y]')

# Gaussian: η = μ (identity link)
axes[1].plot(eta, eta, color='#10b981', linewidth=2)
axes[1].set_title('Gaussian: μ = η (identity link)', color='#e2e8f0')
axes[1].set_xlabel('Natural parameter η')
axes[1].set_ylabel('Mean E[y]')

# Poisson: η = log(λ), so λ = exp(η)
axes[2].plot(eta, np.exp(eta), color='#f59e0b', linewidth=2)
axes[2].set_title('Poisson: λ = exp(η) (log link)', color='#e2e8f0')
axes[2].set_xlabel('Natural parameter η')
axes[2].set_ylabel('Mean E[y]')
axes[2].set_ylim(0, 20)

plt.suptitle('GLM Link Functions: η → E[y]', color='#e2e8f0', y=1.02)
plt.tight_layout()
plt.show()

**What to notice:** the three panels are the same natural parameter `η` mapped through three
**link** functions — sigmoid (Bernoulli), identity (Gaussian), exp (Poisson). Choosing the link is
choosing which distribution your target follows; the linear part `w·x` is shared. That's the unifying
idea of GLMs.

## 2. GLM gradient update — unified form

For any GLM, the gradient of the log-likelihood is:
$$\nabla_\theta \ell = X^\top(y - \hat{y})$$
This is true for linear regression, logistic regression, and Poisson regression — the same update rule.

In [ ]:
def glm_gradient_descent(X, y, link, n_iters=500, lr=0.1):
    """Generic GLM gradient descent. link maps linear predictor → mean."""
    theta = np.zeros(X.shape[1])
    losses = []
    for _ in range(n_iters):
        eta = X @ theta
        y_hat = link(eta)
        grad = X.T @ (y - y_hat)          # universal GLM gradient
        theta += lr * grad / len(y)
        losses.append(np.mean((y - y_hat)**2))
    return theta, losses

# Poisson regression example: predict count of customer tickets
n = 200
X_raw = np.random.randn(n, 2)
X = np.column_stack([np.ones(n), X_raw])
true_theta = np.array([1.0, 0.8, -0.5])
lam_true = np.exp(X @ true_theta)
y_count = np.random.poisson(lam_true)

theta_pois, losses_pois = glm_gradient_descent(X, y_count, link=np.exp, lr=0.01, n_iters=1000)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(losses_pois, color='#f59e0b', linewidth=2)
ax1.set_xlabel('Iteration')
ax1.set_ylabel('MSE')
ax1.set_title('Poisson GLM Training Loss', color='#e2e8f0')

y_pred_pois = np.exp(X @ theta_pois)
ax2.scatter(lam_true, y_pred_pois, alpha=0.4, s=15, color='#f59e0b')
ax2.plot([0, lam_true.max()], [0, lam_true.max()], 'w--', linewidth=1, alpha=0.5)
ax2.set_xlabel('True λ')
ax2.set_ylabel('Predicted λ')
ax2.set_title('Predicted vs True Rate (Poisson GLM)', color='#e2e8f0')
plt.tight_layout()
plt.show()

print(f"True θ:      {true_theta}")
print(f"Fitted θ:    {theta_pois.round(3)}")

**What to notice:** the **same** update `Xᵀ(y − ŷ)` — just with a different link inside `ŷ` —
trains the Poisson model, and the predicted rates track the true `λ` along the diagonal. Swap the
link and this identical code becomes linear regression (identity) or logistic regression (sigmoid).
One optimizer, a family of models.

## 3. Gaussian Discriminant Analysis (GDA)

GDA models the class-conditional distributions $p(x|y)$ as Gaussians, then applies Bayes' theorem.

In [ ]:
# Generate a 2D binary classification dataset
mu0_true, mu1_true = np.array([-2.0, 0.0]), np.array([2.0, 0.0])
Sigma_true = np.array([[1.5, 0.5], [0.5, 1.0]])

n0, n1 = 100, 100
X0 = np.random.multivariate_normal(mu0_true, Sigma_true, n0)
X1 = np.random.multivariate_normal(mu1_true, Sigma_true, n1)
X_data = np.vstack([X0, X1])
y_data = np.array([0]*n0 + [1]*n1)

# --- GDA fit (closed-form MLE) ---
phi = np.mean(y_data)  # P(y=1)
mu0 = X_data[y_data == 0].mean(axis=0)
mu1 = X_data[y_data == 1].mean(axis=0)
# Pooled covariance (equal covariance assumption)
diff0 = X_data[y_data == 0] - mu0
diff1 = X_data[y_data == 1] - mu1
Sigma_hat = (diff0.T @ diff0 + diff1.T @ diff1) / len(y_data)

def gda_predict_proba(X, phi, mu0, mu1, Sigma):
    """P(y=1|x) via Bayes' theorem with Gaussian class-conditionals."""
    from scipy.stats import multivariate_normal
    log_p0 = multivariate_normal.logpdf(X, mu0, Sigma) + np.log(1 - phi)
    log_p1 = multivariate_normal.logpdf(X, mu1, Sigma) + np.log(phi)
    # log-sum-exp for numerical stability
    log_total = np.logaddexp(log_p0, log_p1)
    return np.exp(log_p1 - log_total)

# --- Logistic regression fit ---
Xb = np.column_stack([np.ones(len(X_data)), X_data])
theta_lr, _ = glm_gradient_descent(Xb, y_data, sigmoid, n_iters=2000, lr=0.5)

# --- Plot both decision boundaries ---
xx, yy = np.meshgrid(np.linspace(-6, 6, 200), np.linspace(-4, 4, 200))
grid = np.c_[xx.ravel(), yy.ravel()]

proba_gda = gda_predict_proba(grid, phi, mu0, mu1, Sigma_hat).reshape(xx.shape)
Xb_grid = np.column_stack([np.ones(len(grid)), grid])
proba_lr  = sigmoid(Xb_grid @ theta_lr).reshape(xx.shape)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, proba, title in zip(axes,
                             [proba_gda, proba_lr],
                             ['GDA Decision Boundary', 'Logistic Regression Boundary']):
    ax.contourf(xx, yy, proba, levels=20, cmap='RdBu', alpha=0.6)
    ax.contour(xx, yy, proba, levels=[0.5], colors='white', linewidths=2)
    ax.scatter(X0[:, 0], X0[:, 1], c='#22d3ee', s=15, alpha=0.7, label='Class 0')
    ax.scatter(X1[:, 0], X1[:, 1], c='#f43f5e', s=15, alpha=0.7, label='Class 1')
    ax.set_title(title, color='#e2e8f0')
    ax.legend()

plt.suptitle('GDA vs Logistic Regression — Both Learn a Linear Boundary', color='#e2e8f0')
plt.tight_layout()
plt.show()

print(f"GDA fitted μ₀: {mu0.round(3)}, μ₁: {mu1.round(3)}")
print(f"GDA fitted Σ diag: {np.diag(Sigma_hat).round(3)}")

**What to notice:** GDA (generative — model each class as a Gaussian, then apply Bayes) and
logistic regression (discriminative — model `P(y|x)` directly) produce **nearly the same linear
boundary** here. That's not a coincidence: with a **shared** covariance, GDA's boundary is provably
linear, and it implies the logistic form. They're two routes to a similar classifier.

## The library way — validate against `sklearn`

`sklearn.PoissonRegressor` fits the same Poisson GLM, and `LinearDiscriminantAnalysis` is exactly
GDA with a shared covariance. The cell checks our Poisson coefficients recover the truth and that our
GDA agrees with `sklearn`'s LDA.

In [ ]:
from sklearn.linear_model import PoissonRegressor
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

# Poisson GLM: our GD coefficients vs the truth and sklearn
pr = PoissonRegressor(alpha=1e-8, fit_intercept=False).fit(X, y_count)   # X already has a bias column
print('true theta   :', true_theta)
print('our GD theta :', theta_pois.round(3))
print('sklearn theta:', pr.coef_.round(3))
assert np.allclose(theta_pois, true_theta, atol=0.3), "Poisson GLM must recover the true coefficients"

# GDA == LDA (shared covariance): the two should agree on almost every prediction
gda_pred = (gda_predict_proba(X_data, phi, mu0, mu1, Sigma_hat) >= 0.5).astype(int)
lda_pred = LinearDiscriminantAnalysis().fit(X_data, y_data).predict(X_data)
agree = np.mean(gda_pred == lda_pred)
print(f'\nGDA vs sklearn LDA agreement: {agree:.3f}')
assert agree > 0.95, "our GDA must match sklearn's LDA"
print('Poisson GLM recovers the truth, and our GDA == sklearn LDA ✓')

**What to notice:** our from-scratch Poisson GLM recovers the true coefficients (and matches
`sklearn.PoissonRegressor`), and our hand-built GDA agrees with `sklearn`'s LDA on essentially every
point — confirming GDA-with-shared-covariance *is* LDA. The unified GLM machinery is exactly what the
library implements.

## Gotchas & tradeoffs

- **Match the link to the response.** Counts need a log link (Poisson), not identity — using linear
  regression on counts can predict negatives and mis-model the variance.
- **GDA makes strong assumptions.** It assumes each class is Gaussian; if that's wrong, logistic
  regression (which assumes less) is usually safer.
- **Shared vs per-class covariance.** Shared covariance → **linear** boundary (LDA); per-class →
  **quadratic** (QDA, the exercise below). More flexibility, more parameters to estimate.
- **Generative vs discriminative.** GDA/LDA can be more data-efficient *when its assumptions hold*;
  logistic regression is more robust when they don't — a classic tradeoff.

In [ ]:
# Same linear model, wrong link: fitting counts with identity vs log link
theta_id,  _ = glm_gradient_descent(X, y_count, link=lambda e: e,      lr=0.005, n_iters=1000)
theta_log, _ = glm_gradient_descent(X, y_count, link=np.exp,           lr=0.010, n_iters=1000)
pred_id  = X @ theta_id
print('identity link: fraction of NEGATIVE predicted counts =', np.mean(pred_id < 0).round(3),
      '(impossible for counts!)')
print('log link     : fraction of negative predicted rates   =', np.mean(np.exp(X @ theta_log) < 0).round(3))
print('\n-> the log link guarantees positive rates; the identity link does not')

**What to notice:** fitting count data with an **identity** link predicts a chunk of **negative
counts** (nonsensical), while the **log** link `λ = exp(η)` guarantees positivity by construction.
Choosing the link that matches the response distribution isn't cosmetic — it keeps predictions
valid.

## Key takeaways

- A **GLM** = linear predictor `w·x` + a **link function** matched to the response distribution
  (identity/Gaussian, logit/Bernoulli, log/Poisson).
- The training gradient **`Xᵀ(y − ŷ)`** is the *same* for all GLMs — one optimizer, a family of
  models.
- **GDA** is generative (Gaussian per class); with shared covariance it's **LDA** (linear boundary),
  with per-class it's **QDA** (quadratic).
- **Generative vs discriminative**: GDA is data-efficient when its Gaussian assumption holds;
  logistic regression is more robust otherwise. Validate with `PoissonRegressor` / `LDA`.

**Next:** the [course quiz](https://ml-viz-ruby.vercel.app/courses/linear-regression/05-quiz).

## ✏️ Your turn

**Exercise 1 — Quadratic Discriminant Analysis (QDA).** GDA assumes equal covariance $\Sigma$ for both classes. QDA relaxes this, fitting a separate $\Sigma_0$ and $\Sigma_1$ per class. Implement QDA by fitting separate covariances and observe how the decision boundary becomes non-linear (quadratic).

In [ ]:
# TODO(you): implement QDA
# Sigma0 = covariance of class-0 samples only
# Sigma1 = covariance of class-1 samples only
# Then use gda_predict_proba with Sigma0/Sigma1 independently
# Hint: call gda_predict_proba but modify it to accept per-class covariances

In [ ]:
# Assert cell — passes silently when correct
# Fit QDA with separate covariances
Sigma0_ref = (diff0.T @ diff0) / n0
Sigma1_ref = (diff1.T @ diff1) / n1
print("QDA Σ₀ diag:", np.diag(Sigma0_ref).round(3))
print("QDA Σ₁ diag:", np.diag(Sigma1_ref).round(3))
# Both should be close to Sigma_true diag = [1.5, 1.0]
assert abs(np.diag(Sigma0_ref)[0] - 1.5) < 0.5, "Σ₀[0,0] should be ~1.5"
assert abs(np.diag(Sigma1_ref)[0] - 1.5) < 0.5, "Σ₁[0,0] should be ~1.5 too (both classes share Σ_true)"

# QDA's curvature should make it disagree with GDA when the two classes have genuinely different
# covariances -- build a small dedicated example where that's true (this notebook's own dataset above
# uses equal covariances by construction, so GDA and QDA would nearly agree there)
from scipy.stats import multivariate_normal as _mvn
rng_qda = np.random.default_rng(0)
mu0_q, mu1_q = np.array([-1.0, 0.0]), np.array([1.0, 0.0])
Sigma0_true_q = np.array([[0.2, 0.0], [0.0, 0.2]])  # tight class
Sigma1_true_q = np.array([[2.0, 0.0], [0.0, 2.0]])  # spread-out class
Xq0 = rng_qda.multivariate_normal(mu0_q, Sigma0_true_q, 200)
Xq1 = rng_qda.multivariate_normal(mu1_q, Sigma1_true_q, 200)
Xq = np.vstack([Xq0, Xq1])
yq = np.array([0] * 200 + [1] * 200)
phi_q = np.mean(yq)
mu0_qe, mu1_qe = Xq[yq == 0].mean(axis=0), Xq[yq == 1].mean(axis=0)
diffq0, diffq1 = Xq[yq == 0] - mu0_qe, Xq[yq == 1] - mu1_qe
Sigma0_q, Sigma1_q = (diffq0.T @ diffq0) / 200, (diffq1.T @ diffq1) / 200
Sigma_pooled_q = (diffq0.T @ diffq0 + diffq1.T @ diffq1) / len(yq)

def qda_predict_proba(X, phi, mu0, mu1, Sigma0, Sigma1):
    log_p0 = _mvn.logpdf(X, mu0, Sigma0) + np.log(1 - phi)
    log_p1 = _mvn.logpdf(X, mu1, Sigma1) + np.log(phi)
    log_total = np.logaddexp(log_p0, log_p1)
    return np.exp(log_p1 - log_total)

test_pt = np.array([[-4.0, -4.0]])  # far in the tight class's low-density tail
p_qda = qda_predict_proba(test_pt, phi_q, mu0_qe, mu1_qe, Sigma0_q, Sigma1_q)
p_gda = gda_predict_proba(test_pt, phi_q, mu0_qe, mu1_qe, Sigma_pooled_q)
assert abs(p_qda - p_gda) > 0.3, "QDA's quadratic boundary must disagree sharply with GDA's linear one when covariances genuinely differ"

# Edge case: near-singular per-class covariance -- a class with only 2 (near-)identical points in 2D
# gives a rank-deficient sample covariance, which a naive Gaussian pdf can't invert without regularizing
X_tiny_class = np.array([[1.0, 1.0], [1.0 + 1e-9, 1.0 - 1e-9]])
mu_tiny = X_tiny_class.mean(axis=0)
Sigma_tiny_raw = ((X_tiny_class - mu_tiny).T @ (X_tiny_class - mu_tiny)) / len(X_tiny_class)
raised = False
try:
    _mvn.logpdf(test_pt, mu_tiny, Sigma_tiny_raw)
except np.linalg.LinAlgError:
    raised = True
assert raised, "an (almost) singular per-class covariance should fail without regularization"
eps = 1e-6
Sigma_tiny_reg = Sigma_tiny_raw + eps * np.eye(2)
val_reg = _mvn.logpdf(test_pt, mu_tiny, Sigma_tiny_reg)
assert np.isfinite(val_reg), "adding a small jitter (eps * I) to the covariance fixes it"

<details><summary>Solution</summary>

```python
from scipy.stats import multivariate_normal

Sigma0 = (diff0.T @ diff0) / n0
Sigma1 = (diff1.T @ diff1) / n1

def qda_predict_proba(X, phi, mu0, mu1, Sigma0, Sigma1):
    log_p0 = multivariate_normal.logpdf(X, mu0, Sigma0) + np.log(1 - phi)
    log_p1 = multivariate_normal.logpdf(X, mu1, Sigma1) + np.log(phi)
    log_total = np.logaddexp(log_p0, log_p1)
    return np.exp(log_p1 - log_total)

proba_qda = qda_predict_proba(grid, phi, mu0, mu1, Sigma0, Sigma1).reshape(xx.shape)

plt.figure(figsize=(7, 5))
plt.contourf(xx, yy, proba_qda, levels=20, cmap='RdBu', alpha=0.6)
plt.contour(xx, yy, proba_qda, levels=[0.5], colors='white', linewidths=2)
plt.scatter(X0[:, 0], X0[:, 1], c='#22d3ee', s=15, alpha=0.7, label='Class 0')
plt.scatter(X1[:, 0], X1[:, 1], c='#f43f5e', s=15, alpha=0.7, label='Class 1')
plt.title('QDA — Quadratic (Non-linear) Boundary')
plt.legend()
plt.show()
```

QDA's decision boundary is quadratic (an ellipse or hyperbola) because different covariances mean the log-likelihood ratio contains quadratic terms in $x$. GDA's equal-covariance assumption causes those terms to cancel, leaving a linear boundary.

</details>

---
## 🌐 Extra practice — Softmax regression (DML #105)

### Exercise 2 — Softmax regression: the multi-class GLM

Binary logistic regression is the Bernoulli GLM; **softmax regression** is its direct multi-class extension — the categorical-distribution GLM. DML #105's signature folds the bias in internally (like `train_logreg` in the logistic-regression notebook) and trains a full weight matrix `B` (one column per class) with the softmax + cross-entropy gradient $X^\top(P - Y)$, where `Y` is the one-hot label matrix. The checks pin an exact numeric trace and probe the **all-same-class-labels** edge case: with a single class present, softmax should trivially predict that class with probability 1 and near-zero loss from the very first iteration.

In [ ]:
def train_softmaxreg(X, y, learning_rate, iterations):
    """DML #105 signature: multi-class softmax regression trained with gradient descent.

    Returns
    -------
    B : list[list[float]]
        C x (D+1) parameter matrix (one row per class), rounded to 4 decimals
    losses : list[float]
        cross-entropy loss (summed, not averaged) at every iteration, rounded to 4 decimals
    """
    def softmax(z):
        z = z - z.max(axis=1, keepdims=True)  # numerically-stable softmax
        ez = np.exp(z)
        return ez / ez.sum(axis=1, keepdims=True)

    X = np.asarray(X, dtype=float)
    y = np.asarray(y).astype(int)
    C = y.max() + 1  # classes are assumed to start at 0
    Y = np.eye(C)[y]
    Xb = np.hstack([np.ones((X.shape[0], 1)), X])
    B = np.zeros((Xb.shape[1], C))
    losses = []

    for _ in range(iterations):
        # TODO(you): P = softmax(Xb @ B)
        P = ...
        # TODO(you): B -= learning_rate * Xb.T @ (P - Y)   (the same X^T(pred - target) form as logistic regression)
        # TODO(you): loss = -sum(log(P[i, y_i])) for each row i -- append round(loss, 4) to losses
        ...

    return np.round(B.T, 4).tolist(), losses

In [ ]:
# Checks — run me
X_soft = np.array([[0.5, -1.2], [-0.3, 1.1], [0.8, -0.6]])
y_soft = np.array([0, 1, 2])
B_soft, losses_soft = train_softmaxreg(X_soft, y_soft, 0.01, 10)
assert np.allclose(B_soft, [[-0.0011, 0.0145, -0.0921], [0.002, -0.0598, 0.1263], [-0.0009, 0.0453, -0.0342]]), \
    "DML #105 example"
assert losses_soft[:3] == [3.2958, 3.2611, 3.2272], "loss trace must match, rounded to 4 decimals"

# Loss must never increase (full-batch GD on a convex loss with a small learning rate)
assert all(a >= b - 1e-9 for a, b in zip(losses_soft, losses_soft[1:])), "cross-entropy must be non-increasing"

# Edge case: all-same-class labels -- with a single class, softmax always predicts it correctly
X_one_class = np.array([[1.0, 2.0], [3.0, -1.0], [0.5, 0.5]])
y_one_class = np.array([0, 0, 0])
B_one, losses_one = train_softmaxreg(X_one_class, y_one_class, 0.1, 5)
assert np.all(np.isfinite(np.array(B_one))), "must stay finite with a single class"
assert all(l < 1e-6 for l in losses_one), "a single class is always predicted with probability 1 -> ~0 loss"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def train_softmaxreg(X, y, learning_rate, iterations):
    def softmax(z):
        z = z - z.max(axis=1, keepdims=True)
        ez = np.exp(z)
        return ez / ez.sum(axis=1, keepdims=True)

    X = np.asarray(X, dtype=float)
    y = np.asarray(y).astype(int)
    C = y.max() + 1
    Y = np.eye(C)[y]
    Xb = np.hstack([np.ones((X.shape[0], 1)), X])
    B = np.zeros((Xb.shape[1], C))
    losses = []

    for _ in range(iterations):
        P = softmax(Xb @ B)
        B -= learning_rate * Xb.T @ (P - Y)
        loss = -np.sum(np.log(np.clip(P[np.arange(len(P)), y], 1e-12, None)))
        losses.append(round(float(loss), 4))

    return np.round(B.T, 4).tolist(), losses
```

</details>